# Bài toán 2: Dự đoán Bàn thắng Kì vọng (xG Model)

In [ ]:
!pip install -q mlflow==3.16.1 pysocks kaggle xgboost shap duckdb

In [ ]:
# Upload file colab_utils.py từ máy tính
from google.colab import files
uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

import importlib
import colab_utils
importlib.invalidate_caches()
importlib.reload(colab_utils)
print("File đang dùng:", colab_utils.__file__)


In [ ]:
import colab_utils
from google.colab import userdata

colab_utils.install_tailscale()
colab_utils.setup_tailscale(auth_key=userdata.get('TAILSCALE_AUTH_KEY'))
colab_utils.setup_mlflow("http://20.41.113.183:8080", experiment="xG-Prediction")


In [ ]:
import duckdb
import pandas as pd

con = duckdb.connect(':memory:')
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"""
    CREATE OR REPLACE SECRET minio_secret (
        TYPE S3, KEY_ID '{userdata.get('MINIO_ACCESS_KEY')}', SECRET '{userdata.get('MINIO_SECRET_KEY')}',
        ENDPOINT '20.41.113.183:9000', URL_STYLE 'path', USE_SSL 'false'
    );
""")

# Dữ liệu ~100k cú sút, tỷ lệ bàn thắng ~10%
df_shots = con.execute("SELECT * FROM read_parquet('s3://football-lake/dwh/mart_ml_shots.parquet')").df()
print("Phân phối nhãn:", df_shots['y_is_goal'].value_counts())
display(df_shots.head())


In [ ]:
# Binary Classification
features = [c for c in df_shots.columns if c.startswith('f_') and pd.api.types.is_numeric_dtype(df_shots[c])]
df_shots = df_shots.dropna(subset=features + ['y_is_goal'])

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df_shots[features], df_shots['y_is_goal'], test_size=0.2, random_state=42)

import xgboost as xgb
from sklearn.metrics import roc_auc_score, classification_report
import mlflow
import mlflow.xgboost

# Thiết lập siêu tham số
params = {
    'objective': 'binary:logistic',
    'scale_pos_weight': 9, # vì tỷ lệ 90% trượt, 10% vào
    'learning_rate': 0.05,
    'max_depth': 5,
    'n_estimators': 300,
    'random_state': 42
}

with mlflow.start_run(run_name="xg_model_binary"):
    # 1. Log params
    mlflow.log_params(params)
    mlflow.log_param("n_features", len(features))
    
    # 2. Huấn luyện
    model = xgb.XGBClassifier(**params)
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    
    # 3. Đánh giá (Metrics)
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]
    
    auc = roc_auc_score(y_test, probs)
    print(f"ROC-AUC: {auc:.4f}")
    print("\nClassification Report:\n", classification_report(y_test, preds))
    
    mlflow.log_metrics({
        "test_roc_auc": auc
    })
    
    # 4. Lưu Model vào MLflow
    model_info = mlflow.xgboost.log_model(model, artifact_path="model")
    print(f"Đã lưu mô hình tại URI: {model_info.model_uri}")
    
# ==========================================
# 5. TẢI MÔ HÌNH VỀ TỪ MLFLOW ĐỂ DÙNG THỬ
# ==========================================
print("\n--- Đang tải lại mô hình từ MLflow... ---")
loaded_xg_model = mlflow.xgboost.load_model(model_info.model_uri)

sample_data = X_test.head(5)
sample_labels = y_test.head(5)

print("Dự đoán xG (Expected Goals) cho 5 cú sút đầu tiên trong tập Test:")
pred_xg = loaded_xg_model.predict_proba(sample_data)[:, 1]

for i in range(5):
    print(f"Cú sút {i+1}: Thực tế = {'VÀO' if sample_labels.iloc[i]==1 else 'TRƯỢT'} | xG dự đoán = {pred_xg[i]:.2%}")
